# Lekce 13 - Paměť agenta


## Nastavení

Tento notebook ukazuje, jak vytvořit agenta pro rezervaci cestování s **perzistentní pamětí** pomocí **Microsoft Agent Framework** (MAF).

Naučíte se, jak různé typy paměti agenta — pracovní, krátkodobá a dlouhodobá — ovlivňují, jak agent uchovává a používá informace během konverzací.

**Požadavky:**
- Projekt Azure AI Foundry s nasazeným modelem chatu (např. `gpt-4o-mini`).
- Přihlášený pomocí Azure CLI — spusťte v terminálu příkaz `az login`.
- `AZURE_AI_PROJECT_ENDPOINT` — váš koncový bod projektu Azure AI Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — název vašeho nasazeného modelu.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Typy paměti agenta

AI agenti mohou využívat různé druhy paměti, přičemž každá slouží odlišnému účelu:

### Pracovní paměť
Samotná vlákno konverzace — zprávy vyměněné během jedné relace. Agent se může odkazovat na předchozí zprávy ve stejném vlákně, aby udržel koherenci. V MAF je toto vytvořeno pomocí **`agent.create_session()`**, která vrací `AgentSession`.

### Krátkodobá paměť
Informace, které přetrvávají během trvání úkolu nebo relace, ale nejsou uchovávány trvale. Například agent může během plánovací konverzace s více koly nasbírat fakta a použít je k vytvoření finálního itineráře.

### Dlouhodobá paměť
Preference a fakta, která přetrvávají **mezi relacemi**. Vracející se uživatel by neměl muset opakovat svá dietní omezení nebo cestovní styl. Dlouhodobá paměť je obvykle podporována externím úložištěm — databází, souborem nebo vektorovým indexem — a je agentovi zpřístupněna přes nástroje.


## Pracovní paměť se sessiony

Nejjednodušší forma paměti je konverzační session. Když předáte stejnou session (vytvořenou pomocí `agent.create_session()`) do po sobě jdoucích volání `agent.run()`, agent uvidí celou historii toho rozhovoru a může si vybavit dřívější detaily.

Vytvořme si cestovního agenta a ukažme pracovní paměť.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agent si správně vybavil rozpočet, protože obě zprávy sdílejí stejnou relaci. Toto je **pracovní paměť** — existuje pouze po dobu trvání relace.

### Co se stane s novým vláknem?

Pokud vytvoříme **novou** relaci, agent si nepamatuje předchozí konverzaci:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Vzor dlouhodobé paměti

Pro zapamatování uživatelských preferencí **napříč relacemi** potřebujeme trvalé uložiště, které existuje mimo vlákno konverzace. Agent k tomuto uložišti přistupuje pomocí **nástrojů** — funkcí, které může volat pro ukládání a získávání informací.

Níže implementujeme jednoduché ukládání preferencí v paměti (v produkci byste to podložili databází nebo vektorovým indexem) a zpřístupníme to jako nástroje, které může agent používat.

### Architektura
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scénář 1 — Uživateľ rezervuje výlet k výročí poprvé

Sarah navštíví poprvé. Agent by měl uložit její preference pomocí nástrojů a použít je k doporučení hotelů.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scénář 2 — Sarah se vrací o několik týdnů později

Sarah zahajuje **zbrusu nové vlákno** (simuluje novou relaci). Pracovní paměť je prázdná, ale dlouhodobé úložiště preferencí stále obsahuje její informace. Agent by je měl získat a použít k personalizaci doporučení.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Shrnutí

V této lekci jste se naučili tři typy paměti agenta a jak je implementovat pomocí Microsoft Agent Framework:

| Typ paměti | Mechanismus MAF | Doba trvání |
|---|---|---|
| **Pracovní** | `agent.create_session()` | Jednotlivý rozhovor |
| **Krátkodobá** | Akumulovaný kontext v rámci vlákna | Jednotlivý úkol / relace |
| **Dlouhodobá** | Externí úložiště přistupované přes funkce `@tool` | Napříč relacemi |

### Klíčové poznatky
1. **`agent.create_session()`** poskytuje pracovní paměť — agent vidí celou historii konverzace v rámci relace.
2. **Nové relace ztrácejí kontext** — bez dlouhodobé paměti si agent nemůže vybavit minulé konverzace.
3. **Funkce `@tool`** překlenou mezeru — umožňují agentovi ukládat a načítat informace z perzistentního úložiště.
4. **Personalizace se zlepšuje v čase** — čím více preferencí je uložených, tím lepší jsou doporučení agenta.

### Aplikace v reálném světě
- **Zákaznický servis**: Zapamatovat si historii a preference zákazníka
- **Osobní asistenti**: Udržovat kontext napříč dny nebo týdny
- **Zdravotnictví**: Sledovat informace a preference pacienta
- **E-commerce**: Personalizovaný nákup na základě historie

### Další kroky
- Nahradit in-memory slovník databází nebo vektorovým uložištěm (např. Azure AI Search)
- Přidat expiraci paměti pro časově citlivé informace
- Vytvořit víceagentní systémy se sdílenou pamětí
- Prozkoumat Cognee notebook pro paměť založenou na znalostním grafu


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o vyloučení odpovědnosti**:  
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o přesnost, mějte na paměti, že automatizované překlady mohou obsahovat chyby či nepřesnosti. Původní dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědni za jakékoliv nedorozumění nebo nesprávné interpretace vyplývající z použití tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
